In [5]:
from utils import * 
import subprocess
import json
from Bio.Align import PairwiseAligner 
import itertools
import requests

%load_ext autoreload 
%autoreload 2

ALPHAFOLD_DIR = '../data/genes/alphafold/'
ALPHAFOLD_INPUT_DIR = '../data/genes/alphafold/af_input/'
ALPHAFOLD_OUTPUT_DIR = '../data/genes/alphafold/af_output/'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
def get_msas_unpaired(gene_ids:list, dir_path:str='../data/genes/mmseqs'):
    '''Load the unpaired MSAs for the specified gene IDs.
    
    :param gene_ids: The IDs of the genes to construct paired MSAs for. These should all be from the same genome.   
    :param dir_path: The path to the directory where the MSAs are stored. This function assumes the a3m files have names matching the gene ID of the query 
        sequence. 
    '''
    paths = {gene_id:os.path.join(dir_path, f'{gene_id}.a3m') for gene_id in gene_ids}
    return {gene_id:str(FASTAFile.from_file(path)) for gene_id, path in paths.items()}


In [6]:
genes_df = pd.read_csv('../data/genes/genes.csv', index_col=0)
genes_df['cluster_id'] = genes_df.index.map(json.load(open('../data/genes/clusters.json', 'r')))
genes_df = genes_df[genes_df.cluster_id == 7].copy()
genes_df['alphafold_id'] = genes_df.genome_id.apply(lambda genome_id : f'{genome_id}_cluster_7_1mer')

FASTAFile.from_df(genes_df).write('../data/genes/cluster_7.faa')
# subprocess.run('muscle -align ../data/genes/cluster_1.faa -output ../data/genes/cluster_1.afa', shell=True, check=True)
for row in genes_df.itertuples():
    print(f'{row.Index}')

print('\nMinimum cluster_7 gene length:', genes_df.seq.apply(len).min())
print('Maximum cluster_7 gene length:', genes_df.seq.apply(len).max())

orfm.bz_0.1_82
orfm.bz_1.1_43
orfm.bz_2.1_117
orfm.bz_3.1_142
orfm.bz_5.1_26
orfm.bz_7.1_68
orfm.bz_8.1_106
orfm.bz_9.1_152
orfm.bz_10.1_110
orfm.bz_11.1_147

Minimum cluster_7 gene length: 207
Maximum cluster_7 gene length: 311


In [7]:
tmhmm_file = TMHMMFile.from_file('../data/genes/tmhmm/genes.txt')
genes_df['num_tmhs'] = genes_df.index.map(tmhmm_file.get_num_tmhs())
    
for row in genes_df.itertuples():
    print(f'Number TMHs in {row.genome_id} cluster_1 protein:', row.num_tmhs)


Number TMHs in bz_0 cluster_1 protein: 0
Number TMHs in bz_1 cluster_1 protein: 0
Number TMHs in bz_2 cluster_1 protein: 0
Number TMHs in bz_3 cluster_1 protein: 0
Number TMHs in bz_5 cluster_1 protein: 0
Number TMHs in bz_7 cluster_1 protein: 0
Number TMHs in bz_8 cluster_1 protein: 0
Number TMHs in bz_9 cluster_1 protein: 0
Number TMHs in bz_10 cluster_1 protein: 0
Number TMHs in bz_11 cluster_1 protein: 0


In [13]:
# Because the structure confidence is so low, I am going to re-run the monomer structure predictions on the AlphaFold server

path = os.path.join(ALPHAFOLD_INPUT_DIR, 'cluster_7.json')

inputs = list()
for row in genes_df.itertuples():

    msas = dict()
    msas['unpaired'] = get_msas_unpaired([row.Index])
    input = AlphaFoldInputFile(row.alphafold_id, dialect='alphafoldserver', num_seeds=1, version=2)
    input.add_seq(row.seq, paired_msa='', unpaired_msa=msas['unpaired'][row.Index], n=1)
    inputs.append(input)

with open(path, 'w') as f:
    json.dump([input.get_info() for input in inputs], f)

output_dirs = [os.path.join(ALPHAFOLD_DIR, 'fold_' + input.name.replace('-', '_')) for input in inputs]
print('\n'.join(output_dirs))
# outputs = [AlphaFoldServerOutput(path) for path in output_dirs if os.path.exists(path)]

../data/genes/alphafold/fold_bz_0_cluster_7_1mer
../data/genes/alphafold/fold_bz_1_cluster_7_1mer
../data/genes/alphafold/fold_bz_2_cluster_7_1mer
../data/genes/alphafold/fold_bz_3_cluster_7_1mer
../data/genes/alphafold/fold_bz_5_cluster_7_1mer
../data/genes/alphafold/fold_bz_7_cluster_7_1mer
../data/genes/alphafold/fold_bz_8_cluster_7_1mer
../data/genes/alphafold/fold_bz_9_cluster_7_1mer
../data/genes/alphafold/fold_bz_10_cluster_7_1mer
../data/genes/alphafold/fold_bz_11_cluster_7_1mer
